### Comparaison des Modélisations pour la Fréquence et les Coûts des Sinistres

Dans cette section, nous allons comparer différentes approches de modélisation afin d'évaluer deux aspects clés des sinistres :

1. **La fréquence des sinistres** : Modélisation du nombre de sinistres survenus pour chaque contrat.
2. **Les coûts des sinistres** : Estimation des montants associés aux sinistres.

L'objectif est d'identifier les modèles les plus performants pour chaque aspect, en utilisant des métriques d'évaluation adaptées.

In [28]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
import re
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm


In [2]:
freq = pd.read_parquet("data/raw/freMTPLfreq.parquet")
sev = pd.read_parquet("data/raw/freMTPLsev.parquet")

In [3]:
#Combiner les 2 bases de données

freq['PolicyID'] = freq['PolicyID'].astype(str)
sev['PolicyID'] = sev['PolicyID'].astype(str)

# Agrégation de sev : un contrat peut avoir plusieurs sinistres
sev_agg = (
    sev.groupby("PolicyID")
       .agg(
           ClaimAmount=("ClaimAmount", "sum"),   # coût total des sinistres du contrat
           ClaimNb_sev=("ClaimAmount", "size")   # nombre de sinistres déclarés dans sev
       )
       .reset_index()
)

print(len(sev), "lignes sev brutes")
print(len(sev_agg), "lignes sev agrégées (1 par PolicyID)")

merged = pd.merge(freq, sev_agg, on="PolicyID", how="left")

# Remplacer les NaN (contrats sans sinistre) par 0
merged["ClaimAmount"] = merged["ClaimAmount"].fillna(0)
merged["ClaimNb_sev"] = merged["ClaimNb_sev"].fillna(0)

print("Nb de lignes freq :", len(freq))
print("Nb de lignes merged :", len(merged))


16181 lignes sev brutes
15390 lignes sev agrégées (1 par PolicyID)
Nb de lignes freq : 413169
Nb de lignes merged : 413169


# Création de différentes classes actuarielles nécessaires pour notre modélisation


In [4]:
# 1) Contrôles de base
print(merged[["PolicyID","ClaimNb","Exposure","ClaimAmount"]].head())
print(merged[["ClaimNb","Exposure","ClaimAmount"]].describe())

# 2) Variables actuarielles de base
merged["Freq"] = merged["ClaimNb"] / merged["Exposure"]          # fréquence annuelle
merged["PurePremium"] = merged["ClaimAmount"] / merged["Exposure"]  # prime pure
merged["AvgClaim"] = np.where(                                    # coût moyen conditionnel
    merged["ClaimNb"] > 0,
    merged["ClaimAmount"] / merged["ClaimNb"],
    np.nan
)

# 3) Quelques ratios globaux (à commenter dans le rapport)
tot_expo = merged["Exposure"].sum()
tot_claims = merged["ClaimNb"].sum()
tot_amount = merged["ClaimAmount"].sum()

freq_globale = tot_claims / tot_expo
pure_prem_globale = tot_amount / tot_expo
avgclaim_globale = tot_amount / tot_claims

print("Fréquence globale :", freq_globale)
print("Prime pure globale :", pure_prem_globale)
print("Coût moyen global :", avgclaim_globale)
print(f"% contrats sans sinistre: {(merged['ClaimNb']==0).mean()*100:.1f}%")

merged.columns

  PolicyID  ClaimNb  Exposure  ClaimAmount
0        1        0      0.09          0.0
1        2        0      0.84          0.0
2        3        0      0.52          0.0
3        4        0      0.45          0.0
4        5        0      0.15          0.0
             ClaimNb       Exposure   ClaimAmount
count  413169.000000  413169.000000  4.131690e+05
mean        0.039163       0.561088  8.341642e+01
std         0.204053       0.369477  4.192526e+03
min         0.000000       0.002732  0.000000e+00
25%         0.000000       0.200000  0.000000e+00
50%         0.000000       0.540000  0.000000e+00
75%         0.000000       1.000000  0.000000e+00
max         4.000000       1.990000  2.036833e+06
Fréquence globale : 0.06979858984933181
Prime pure globale : 148.66904231188673
Coût moyen global : 2129.9720042024596
% contrats sans sinistre: 96.3%


Index(['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density', 'ClaimAmount', 'ClaimNb_sev',
       'Freq', 'PurePremium', 'AvgClaim'],
      dtype='object')

# Premières approches de modélisation avec loi Poisson pour la fréquence et loi Gamma pour la sévérité

In [5]:

# ============================================================
# 1. Préparation des données et création des classes actuarielles
# ============================================================

# On garde uniquement les contrats exposés
df_glm = merged[merged["Exposure"] > 0].copy()

# Variables de classes si besoin
bins_driver = [17, 25, 30, 40, 50, 60, 120]
labels_driver = ["<25", "25-29", "30-39", "40-49", "50-59", "60+"]
df_glm["DriverAgeClass"] = pd.cut(df_glm["DriverAge"], bins=bins_driver, labels=labels_driver)

bins_car = [-1, 1, 5, 10, 20, 200]
labels_car = ["0", "1-4", "5-9", "10-19", "20+"]
df_glm["CarAgeClass"] = pd.cut(df_glm["CarAge"], bins=bins_car, labels=labels_car)

bins_dens = [0, 50, 200, 500, 2000, 10000]
labels_dens = ["rural", "peri-urbain", "petite ville", "ville", "urbain dense"]
df_glm["DensityClass"] = pd.cut(df_glm["Density"], bins=bins_dens, labels=labels_dens)

# Déclaration des qualitatives
for col in ["Power", "Brand", "Gas", "Region",
            "DriverAgeClass", "CarAgeClass", "DensityClass"]:
    df_glm[col] = df_glm[col].astype("category")

print("Taille base GLM :", len(df_glm))


Taille base GLM : 413169


In [22]:
# =========================
# 2. Split 75 % / 25 %
# =========================

df_train, df_test = train_test_split(df_glm, test_size=0.25, random_state=123)

print("Taille train :", len(df_train))
print("Taille test  :", len(df_test))


Taille train : 309876
Taille test  : 103293


In [8]:
# Base des contrats sinistrés
df_gamma_sev_train = df_train[df_train["ClaimNb"] > 0].copy() #on garde que les contrats avec au moins un sinistre
df_gamma_sev_train["AvgClaim"] = df_gamma_sev_train["ClaimAmount"] / df_gamma_sev_train["ClaimNb"]
print("Taille base sévérité (train) :", len(df_gamma_sev_train))

df_gamma_sev_test = df_test[df_test["ClaimNb"] > 0].copy()
df_gamma_sev_test["AvgClaim"] = df_gamma_sev_test["ClaimAmount"] / df_gamma_sev_test["ClaimNb"]
print("Taille base sévérité (test) :", len(df_gamma_sev_test))

Taille base sévérité (train) : 11511
Taille base sévérité (test) : 3879


## Comparaison entre GLM Poisson et GLM Négative Binomiale Basé sur l'AIC et la deviance (pour la fréquence)

In [27]:
# =========================
# 3. Modèle Poisson complet
# =========================

formula_full = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass) + C(Brand)"
)

model_full = smf.glm(
    formula=formula_full,
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train["Exposure"])
)
res_pois = model_full.fit()

aic_poi=res_pois.aic
dev_poi=res_pois.deviance
print("AIC Poisson complet :", res_pois.aic)
print("deviance",res_pois.deviance)


# =========================
# 3. Binomiale négative complète (train)
# =========================

nb_family = sm.families.NegativeBinomial()

model_nb = smf.glm(
    formula=formula_full,
    data=df_train,
    family=nb_family,
    offset=np.log(df_train["Exposure"])
)
res_nb = model_nb.fit()
aic_nb=res_nb.aic
dev_nb=res_nb.deviance
print("AIC Binomiale négative :", res_nb.aic)
print("deviance", res_nb.deviance)

AIC Poisson complet : 96237.59914789078
deviance 73891.03017112348


/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


AIC Binomiale négative : 96019.11212487996
deviance 64970.100145433316


In [ ]:
#Implémentation stepwise qui permet la sélection des variables selon l'AIC: une variable est ajoutée ou supprimée uniquement si cela diminue l’AIC du modèle courant ; sinon l’algorithme s’arrête.

def stepwise_glm_offset(data, response, candidates, family, offset_var, verbose=True):
    """
    data       : DataFrame
    response   : variable cible
    candidates : liste de termes de formule, ex. ["C(Region)", "C(DensityClass)"]
    family     : famille GLM, ex. sm.families.Poisson()
    offset_var : variable d'exposition
    """
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # ----- FORWARD : ajout de la meilleure variable -----
        best_aic = None
        best_var = None

        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.glm(
                formula=formula,
                data=data,
                family=family,
                offset=np.log(data[offset_var])
            ).fit()
            aic = model.aic

            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # ----- BACKWARD : retrait de la pire variable -----
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None

            for var in selected:
                trial_vars = [v for v in selected if v != var]
                rhs = " + ".join(trial_vars) if trial_vars else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.glm(
                    formula=formula,
                    data=data,
                    family=family,
                    offset=np.log(data[offset_var])
                ).fit()
                aic = model.aic

                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var

            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    # ----- modèle final -----
    if selected:
        rhs = " + ".join(selected)
    else:
        rhs = "1"

    final_formula = f"{response} ~ {rhs}"
    final_model = smf.glm(
        formula=final_formula,
        data=data,
        family=family,
        offset=np.log(data[offset_var])
    ).fit()

    if verbose:
        print("Formule finale :", final_formula)

    return final_model, selected


In [29]:
df_poisson_freq_test2 = df_test.copy()
df_poisson_freq_train2 = df_train.copy()

candidates = [
    "C(DriverAgeClass)","C(CarAgeClass) ",
    "C(Power)","C(Region)","C(Gas)","(DensityClass)"
]

family = sm.families.Poisson()

model_step_poisson, vars_selected = stepwise_glm_offset(
    data=df_poisson_freq_train2,
    response="ClaimNb",
    candidates=candidates,
    family=family,
    offset_var="Exposure",
    verbose=True
)

print("Variables retenues :", vars_selected)
print(model_step.summary())


Ajout de (DensityClass), AIC = 97074.75
Ajout de C(DriverAgeClass), AIC = 96494.97
Ajout de C(Gas), AIC = 96370.81
Ajout de C(CarAgeClass) , AIC = 96335.82
Ajout de C(Power), AIC = 96303.19
Ajout de C(Region), AIC = 96302.78
Formule finale : ClaimNb ~ (DensityClass) + C(DriverAgeClass) + C(Gas) + C(CarAgeClass)  + C(Power) + C(Region)
Variables retenues : ['(DensityClass)', 'C(DriverAgeClass)', 'C(Gas)', 'C(CarAgeClass) ', 'C(Power)', 'C(Region)']
                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               296411
Model:                            GLM   Df Residuals:                   296376
Model Family:                 Poisson   Df Model:                           34
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -48116.
Date:                Fri, 05 Dec 2025   Deviance:                       73

In [32]:
aic_poisson=model_step_poisson.aic
dev_poisson=model_step_poisson.deviance
print("AIC Poisson :", aic_poisson)
print("deviance", dev_poisson)

AIC Poisson : 96302.78139129127
deviance 73968.21241452396


In [26]:
candidates = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]

# Famille binomiale négative au lieu de Poisson
family = sm.families.NegativeBinomial()

model_step_nb, vars_selected_nb = stepwise_glm_offset(
    data=df_poisson_freq_train2,
    response="ClaimNb",
    candidates=candidates,
    family=family,
    offset_var="Exposure",
    verbose=True
)

print("Variables retenues (NB) :", vars_selected_nb)
print(model_step_nb.summary())

/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Ajout de C(DensityClass), AIC = 96822.95
Ajout de C(DriverAgeClass), AIC = 96261.05
Ajout de C(Gas), AIC = 96143.73
Ajout de C(CarAgeClass), AIC = 96110.49
Ajout de C(Power), AIC = 96081.05
Formule finale : ClaimNb ~ C(DensityClass) + C(DriverAgeClass) + C(Gas) + C(CarAgeClass) + C(Power)
Variables retenues (NB) : ['C(DensityClass)', 'C(DriverAgeClass)', 'C(Gas)', 'C(CarAgeClass)', 'C(Power)']
                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               296411
Model:                            GLM   Df Residuals:                   296385
Model Family:        NegativeBinomial   Df Model:                           25
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -48015.
Date:                Fri, 05 Dec 2025   Deviance:                       65062.
Time:                        20:30:25   Pearson ch

In [33]:
aic_nb=model_step_nb.aic
dev_nb=model_step_nb.deviance
print("AIC Poisson :", aic_nb)
print("deviance", dev_nb)

AIC Poisson : 96081.05024076189
deviance 65062.03826131525


In [34]:
# =========================
# 6. Modèle retenu
# =========================

if (model_step_nb.aic < model_step_poisson.aic) and (model_step_nb.deviance < model_step_poisson.deviance):
    res_best = model_step_nb
    best_name = "Binomiale négative"
else:
    res_best = model_step_poisson
    best_name = "Poisson"

print("Modèle retenu pour la tarification (comparaison Poisson vs NB) :", best_name)

df_best = df_test.copy()

# Fréquence prédite par contrat avec le modèle retenu
df_best["lambda_hat_best"] = res_best.predict(df_best, offset=np.log(df_best["Exposure"]))

freq_obs = df_best["ClaimNb"].sum() / df_best["Exposure"].sum()
freq_hat_nb = df_best["lambda_hat_best"].sum() / df_best["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite (NB) : {freq_hat_nb:.5f}")

Modèle retenu pour la tarification (comparaison Poisson vs NB) : Binomiale négative
Fréquence observée : 0.07064
Fréquence prédite (NB) : 0.06649



### Modèle retenu pour la tarification : Binomiale négative

Le modèle de fréquence basé sur une distribution **Binomiale négative** a été retenu pour la tarification. Ce choix repose sur une comparaison avec le modèle Poisson, où la Binomiale négative s'est avérée plus adaptée pour gérer la sur-dispersion des données (variance supérieure à la moyenne).

- **Fréquence observée** : 0.07064  
    Cette valeur correspond à la fréquence moyenne des sinistres observés dans les données de test.

- **Fréquence prédite (NB)** : 0.06649  
    Cette valeur représente la fréquence moyenne des sinistres prédite par le modèle Binomiale négative. Bien que légèrement inférieure à la fréquence observée, elle reste cohérente avec les données et reflète la capacité du modèle à capturer les tendances globales.
```

## Comparaison entre GLM gamma et GLM lognormal basé sur l'AIC et la deviance 


In [ ]:
#Implémentation stepwise qui permet la sélection des variables selon l'AIC: une variable est ajoutée ou supprimée uniquement si cela diminue l’AIC du modèle courant ; sinon l’algorithme s’arrête.

def stepwise_glm_offset_gamma(data, response, candidates, family, verbose=True):
    """
    data       : DataFrame
    response   : variable cible
    candidates : liste de termes de formule, ex. ["C(Region)", "C(DensityClass)"]
    family     : famille GLM, ex. sm.families.Poisson()
    offset_var : variable d'exposition
    """
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # ----- FORWARD : ajout de la meilleure variable -----
        best_aic = None
        best_var = None

        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.glm(
                formula=formula,
                data=data,
                family=family
            ).fit()
            aic = model.aic

            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # ----- BACKWARD : retrait de la pire variable -----
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None

            for var in selected:
                trial_vars = [v for v in selected if v != var]
                rhs = " + ".join(trial_vars) if trial_vars else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.glm(
                    formula=formula,
                    data=data,
                    family=family
                ).fit()
                aic = model.aic

                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var

            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    # ----- modèle final -----
    if selected:
        rhs = " + ".join(selected)
    else:
        rhs = "1"

    final_formula = f"{response} ~ {rhs}"
    final_model = smf.glm(
        formula=final_formula,
        data=data,
        family=family
    ).fit()

    if verbose:
        print("Formule finale :", final_formula)

    return final_model, selected



/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


Ajout de C(DriverAgeClass), AIC = 238189.02
Ajout de C(DensityClass), AIC = 225900.28
Ajout de C(Power), AIC = 219956.57
Ajout de C(Region), AIC = 219208.78
Ajout de C(CarAgeClass), AIC = 219051.65
Ajout de C(Gas), AIC = 219030.44
Formule finale : AvgClaim ~ C(DriverAgeClass) + C(DensityClass) + C(Power) + C(Region) + C(CarAgeClass) + C(Gas)
Variables Gamma retenues : ['C(DriverAgeClass)', 'C(DensityClass)', 'C(Power)', 'C(Region)', 'C(CarAgeClass)', 'C(Gas)']
AIC Gamma : 219030.43990804325


In [58]:
df_sev_train_ln = df_gamma_sev_train.copy()
df_sev_test_ln  = df_gamma_sev_test.copy()



candidates_sev = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]

family_gamma = sm.families.Gamma(sm.families.links.log())

model_gamma, vars_gamma = stepwise_glm_offset_gamma(
    data=df_gamma_sev_train,        # DataFrame de sévérité (contrats sinistrés)
    response="AvgClaim",   # sévérité moyenne par sinistre ou par contrat
    candidates=candidates_sev,
    family=family_gamma,
    verbose=True
)

print("Variables Gamma retenues :", vars_gamma)
print("AIC Gamma :", model_gamma.aic)


/opt/python/lib/python3.13/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


Ajout de C(DriverAgeClass), AIC = 238189.02
Ajout de C(DensityClass), AIC = 225900.28
Ajout de C(Power), AIC = 219956.57
Ajout de C(Region), AIC = 219208.78
Ajout de C(CarAgeClass), AIC = 219051.65
Ajout de C(Gas), AIC = 219030.44
Formule finale : AvgClaim ~ C(DriverAgeClass) + C(DensityClass) + C(Power) + C(Region) + C(CarAgeClass) + C(Gas)
Variables Gamma retenues : ['C(DriverAgeClass)', 'C(DensityClass)', 'C(Power)', 'C(Region)', 'C(CarAgeClass)', 'C(Gas)']
AIC Gamma : 219030.43990804325


In [59]:
aic_g=model_gamma.aic
dev_g=model_gamma.deviance
print("AIC Gamma :", aic_g)
print("deviance", dev_g)

AIC Gamma : 219030.43990804325
deviance 16001.546530687665


In [60]:
def stepwise_ols(data, response, candidates, verbose=True):
    selected = []
    remaining = candidates.copy()
    current_aic = None
    changed = True

    while changed:
        changed = False

        # FORWARD
        best_aic = None
        best_var = None
        for var in remaining:
            terms = selected + [var]
            rhs = " + ".join(terms) if terms else "1"
            formula = f"{response} ~ {rhs}"
            model = smf.ols(formula=formula, data=data).fit()
            aic = model.aic
            if (best_aic is None) or (aic < best_aic):
                best_aic = aic
                best_var = var

        if best_var is not None and (current_aic is None or best_aic < current_aic):
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = best_aic
            changed = True
            if verbose:
                print(f"Ajout de {best_var}, AIC = {current_aic:.2f}")

        # BACKWARD
        improved = True
        while improved and len(selected) > 1:
            improved = False
            worst_aic = current_aic
            worst_var = None
            for var in selected:
                trial = [v for v in selected if v != var]
                rhs = " + ".join(trial) if trial else "1"
                formula = f"{response} ~ {rhs}"
                model = smf.ols(formula=formula, data=data).fit()
                aic = model.aic
                if aic < worst_aic:
                    worst_aic = aic
                    worst_var = var
            if worst_var is not None:
                selected.remove(worst_var)
                current_aic = worst_aic
                improved = True
                changed = True
                if verbose:
                    print(f"Retrait de {worst_var}, AIC = {current_aic:.2f}")

    rhs = " + ".join(selected) if selected else "1"
    final_formula = f"{response} ~ {rhs}"
    final_model = smf.ols(formula=final_formula, data=data).fit()
    if verbose:
        print("Formule finale lognormal :", final_formula)
    return final_model, selected



In [62]:
df_sev_train_ln["logAvg"] = np.log(df_sev_train_ln["AvgClaim"])
df_sev_test_ln["logAvg"]  = np.log(df_sev_test_ln["AvgClaim"])


candidates_sev = [
    "C(DriverAgeClass)", "C(CarAgeClass)",
    "C(Power)", "C(Region)", "C(Gas)", "C(DensityClass)"
]


model_logn, vars_logn = stepwise_ols(
    data=df_sev_train_ln,
    response="logAvg",
    candidates=candidates_sev,
    verbose=True
)

print("Variables lognormales retenues :", vars_logn)
print("AIC lognormal (train) :", model_logn.aic)

Ajout de C(DensityClass), AIC = 33299.74
Ajout de C(DriverAgeClass), AIC = 33279.96
Ajout de C(Region), AIC = 33275.26
Formule finale lognormal : logAvg ~ C(DensityClass) + C(DriverAgeClass) + C(Region)
Variables lognormales retenues : ['C(DensityClass)', 'C(DriverAgeClass)', 'C(Region)']
AIC lognormal (train) : 33275.258432621355


In [64]:

# =========================
# 4. Déviance Gamma sur le test (critère commun)
# =========================
print("AIC Gamma :", aic_g)
print("AIC lognormal (train) :", model_logn.aic)

def gamma_deviance(mu, y):
    """Déviance Gamma (scale=1) pour vecteurs numpy positifs."""
    eps = 1e-10
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    mask = (y > 0) & (mu > 0) & ~np.isnan(mu)
    y = y[mask]
    mu = mu[mask]
    return 2 * np.sum((y - mu) / mu - np.log(y / mu + eps)), mask.sum()

# Prédictions Gamma
mu_gamma_test = model_gamma.predict(df_sev_test_ln)
dev_gamma, n_g = gamma_deviance(mu_gamma_test, df_sev_test_ln["AvgClaim"])
print(f"Déviance Gamma (test, n={n_g}) :", dev_gamma)

# Prédictions Lognormal ramenées au niveau moyen (E[Y] ≈ exp(m + s²/2])
mu_logn_test_ln = model_logn.predict(df_sev_test_ln)
sigma2 = model_logn.scale  # variance résiduelle sur log
mu_logn_test = np.exp(mu_logn_test_ln + 0.5 * sigma2)

dev_logn, n_l = gamma_deviance(mu_logn_test, df_sev_test_ln["AvgClaim"])
print(f"Déviance (critère Gamma) Lognormal (test, n={n_l}) :", dev_logn)

# =========================
# 5. Modèle de sévérité retenu
# =========================

if (model_logn.aic < model_gamma.aic) and (dev_logn < dev_gamma):
    sev_best = "Lognormal"
else:
    sev_best = "Gamma"

print("Modèle retenu pour la sévérité :", sev_best)


AIC Gamma : 219030.43990804325
AIC lognormal (train) : 33275.258432621355
Déviance Gamma (test, n=3703) : 6643.9854634817875
Déviance (critère Gamma) Lognormal (test, n=3703) : 6432.920269925579
Modèle retenu pour la sévérité : Lognormal


### Choix du Modèle Lognormal pour la Sévérité

Après comparaison des performances des modèles Gamma et Lognormal pour la modélisation de la sévérité, le modèle **Lognormal** a été retenu. Voici les principales raisons de ce choix :

1. **AIC (Akaike Information Criterion)** :  
    - Le modèle Lognormal présente un AIC significativement plus faible que celui du modèle Gamma, indiquant une meilleure adéquation aux données d'entraînement.

2. **Déviance sur les données de test** :  
    - La déviance du modèle Lognormal est inférieure à celle du modèle Gamma, ce qui montre une meilleure capacité à prédire les données de test.

3. **Performance globale** :  
    - Le modèle Lognormal capture mieux les relations dans les données, ce qui en fait un choix plus robuste pour la modélisation de la sévérité.

En conclusion, le modèle Lognormal est plus performant et sera utilisé pour estimer les coûts moyens conditionnels des sinistres.


In [68]:
res_sev_ln = model_logn
print(res_sev_ln.summary())
print("AIC Lognormal sévérité (train) :", res_sev_ln.aic)

# Sévérité prédite (espérance lognormale) sur train / test
sigma2 = res_sev_ln.scale

df_sev_train_ln["logAvg_hat"] = res_sev_ln.predict(df_sev_train_ln)
df_sev_test_ln["logAvg_hat"]  = res_sev_ln.predict(df_sev_test_ln)

df_sev_train_ln["sev_hat"] = np.exp(df_sev_train_ln["logAvg_hat"] + 0.5 * sigma2)
df_sev_test_ln["sev_hat"]  = np.exp(df_sev_test_ln["logAvg_hat"]  + 0.5 * sigma2)



print(df_sev_test_ln["sev_hat"] .mean())

                            OLS Regression Results                            
Dep. Variable:                 logAvg   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     3.221
Date:                Fri, 05 Dec 2025   Prob (F-statistic):           4.49e-06
Time:                        22:26:11   Log-Likelihood:                -16619.
No. Observations:               10974   AIC:                         3.328e+04
Df Residuals:                   10955   BIC:                         3.341e+04
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

In [69]:
#on rajoute les frequences estimées au dataframe de train et de test

df_train["lambda_hat"] = model_step_nb.predict(df_train, offset=np.log(df_train["Exposure"]))
df_test["lambda_hat"]  = model_step_nb.predict(df_test,  offset=np.log(df_test["Exposure"]))

# Rattacher sev_hat aux dataframes fréquence
df_train = df_train.merge(
    df_sev_train_ln[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)
df_test = df_test.merge(
    df_sev_test_ln[["PolicyID", "sev_hat"]],
    on="PolicyID",
    how="left"
)

# Tarification & validation

In [70]:
# Pour les contrats sans sinistre (pas de prédiction directe), on peut
# utiliser la sévérité moyenne globale estimée sur sev_train
sev_global = df_train["sev_hat"].mean()
df_train["sev_hat"] = df_train["sev_hat"].fillna(sev_global)
df_test["sev_hat"]  = df_test["sev_hat"].fillna(sev_global)

# ============================================================
# 4. Prime pure modélisée et tarif
# ============================================================

# Prime pure modélisée par contrat (unité d'exposition)
df_train["PurePremium_hat"] = df_train["lambda_hat"] * df_train["sev_hat"]
df_test["PurePremium_hat"]  = df_test["lambda_hat"]  * df_test["sev_hat"]

# Chargement global (ex : +20 %)
loading = 1.20
df_train["Tarif"] = df_train["PurePremium_hat"] * loading
df_test["Tarif"]  = df_test["PurePremium_hat"]  * loading

# ============================================================
# 5. Vérification du tarif sur l’échantillon de validation
#    -> comparaison prime pure observée vs modélisée par segment
# ============================================================

group_cols = ["DriverAgeClass", "Power", "Region"]

check_test = (
    df_test
    .groupby(group_cols)
    .agg(
        Exposure_tot=("Exposure", "sum"),
        Pure_obs=("PurePremium", "mean"),
        Pure_hat=("PurePremium_hat", "mean"),
        Tarif_moy=("Tarif", "mean")
    )
    .reset_index()
)

print("Vérification sur l'échantillon de validation (quelques segments) :")
display(check_test.sort_values("Exposure_tot", ascending=False).head(15))

# Comparaison globale sur le test
pure_obs_globale = (df_test["ClaimAmount"].sum() / df_test["Exposure"].sum())
pure_hat_globale = (df_test["PurePremium_hat"].sum() / df_test["Exposure"].sum())

print("Prime pure observée (test)  :", pure_obs_globale)
print("Prime pure modélisée (test) :", pure_hat_globale)

# Ratio modèle / observé
print("Ratio modèle / observé :", pure_hat_globale / pure_obs_globale)

Vérification sur l'échantillon de validation (quelques segments) :


/tmp/ipykernel_58727/1791397941.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_cols)


,DriverAgeClass,Power,Region,Exposure_tot,Pure_obs,Pure_hat,Tarif_moy
383,40-49,f,Centre,1608.721288,323.353863,76.238508,91.486209
263,30-39,f,Centre,1553.444164,305.425258,61.677823,74.013388
393,40-49,g,Centre,1473.794849,182.326372,71.023893,85.228672
623,60+,f,Centre,1446.532909,502.597644,74.459012,89.350815
513,50-59,g,Centre,1372.538418,214.761272,67.946292,81.535550
503,50-59,f,Centre,1306.773798,753.431430,69.252609,83.103130
633,60+,g,Centre,1272.075609,77.940247,69.016772,82.820127
273,30-39,g,Centre,1265.293205,148.026016,54.957077,65.948493
373,40-49,e,Centre,1123.480222,99.148069,77.295212,92.754255
253,30-39,e,Centre,1116.862079,1096.095959,64.263420,77.116104


Prime pure observée (test)  : 148.2704466040269
Prime pure modélisée (test) : 114.32340407841879
Ratio modèle / observé : 0.7710464674307784


## mise en place d'un GLM poisson avec lasso pour la selection de variables


In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import make_scorer, mean_poisson_deviance
import numpy as np

# =========================
# 1. Préparation des données
# =========================

# Les données sont déjà préparées dans X_encoded, y, et sample_weight
# X_encoded : variables explicatives (one-hot encodées)
# y : fréquence (ClaimNb / Exposure)
# sample_weight : poids (Exposure)

# =========================
# 2. Validation croisée pour alpha
# =========================

df = df_glm.copy()
df = df[df["Exposure"] > 0].copy()

# Variable réponse = fréquence
y = df["ClaimNb"] / df["Exposure"]

# Variables explicatives (toutes qualitatives ici)
X_cat = df[["DriverAgeClass", "CarAgeClass", "Power", "Region", "Gas", "DensityClass"]].astype("category")

# One-hot encoding sans drop de la catégorie de base (scikit gère la pénalisation)
enc = OneHotEncoder(drop=None, sparse_output=False)
X_encoded = enc.fit_transform(X_cat)

# Poids = exposition (offset en GLM)
sample_weight = df["Exposure"].to_numpy()

# Split 75 / 25
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_encoded, y, sample_weight, test_size=0.25, random_state=123
)


# Définir une grille de valeurs pour alpha
alpha_grid = np.logspace(-4, 1, 50)

# Définir le modèle Poisson avec Lasso
model = PoissonRegressor(fit_intercept=True, max_iter=1000)

# Définir la validation croisée
cv = KFold(n_splits=5, shuffle=True, random_state=123)

# Définir le score basé sur la déviance de Poisson
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False, needs_proba=True)

# Configurer la recherche de grille
grid_search = GridSearchCV(
    estimator=model,
    param_grid={'alpha': alpha_grid},
    scoring=scorer,
    cv=cv,
    n_jobs=-1
)

# Ajuster le modèle
grid_search.fit(X_encoded, y, sample_weight=sample_weight)

# Meilleur alpha
best_alpha = grid_search.best_params_['alpha']
print("Meilleur alpha (Lasso) :", best_alpha)

# =========================
# 3. Modèle final avec le meilleur alpha
# =========================

model_lasso = PoissonRegressor(alpha=best_alpha, fit_intercept=True, max_iter=1000)
model_lasso.fit(X_encoded, y, sample_weight=sample_weight)

# Récupérer les coefficients non nuls avec leurs variables
coef = model_lasso.coef_
feature_names = enc.get_feature_names_out(X_cat.columns)
# Calculate AIC and Deviance for the model
aic = model_lasso.score(X_encoded, y)
deviance = mean_poisson_deviance(y, model_lasso.predict(X_encoded), sample_weight=sample_weight)

print(f"AIC of the model: {aic:.2f}")
print(f"Deviance of the model: {deviance:.2f}")

selected = [(name, c) for name, c in zip(feature_names, coef) if abs(c) > 1e-6]
selected = sorted(selected, key=lambda x: x[1], reverse=True)

print("Variables sélectionnées par le Lasso (coefs non nuls) :")
for name, c in selected:
    print(f"{name:35s}  coef={c:.3f}")

c:\Users\thoma\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_scorer.py:548: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(
c:\Users\thoma\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1051: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Meilleur alpha (Lasso) : 0.0001
AIC of the model: -0.01
Deviance of the model: 0.45
Variables sélectionnées par le Lasso (coefs non nuls) :
DriverAgeClass_<25                   coef=0.633
DensityClass_nan                     coef=0.332
DensityClass_urbain dense            coef=0.214
Region_Limousin                      coef=0.160
CarAgeClass_5-9                      coef=0.159
Power_k                              coef=0.144
Gas_Diesel                           coef=0.121
Power_i                              coef=0.103
DensityClass_ville                   coef=0.087
CarAgeClass_1-4                      coef=0.081
Region_Poitou-Charentes              coef=0.062
Power_j                              coef=0.050
Power_n                              coef=0.044
CarAgeClass_0                        coef=0.043
Power_m                              coef=0.040
CarAgeClass_10-19                    coef=0.037
Region_Aquitaine                     coef=0.030
DriverAgeClass_25-29                 coef=0.

In [21]:
# 4) Fréquence prédite et contrôle global
lasso_pred = model_lasso.predict(X_test)

freq_obs = df_best["ClaimNb"].sum() / df_best["Exposure"].sum()
freq_hat_nb = lasso_pred.sum() / df_best["Exposure"].sum()
print(f"Fréquence observée : {freq_obs:.5f}")
print(f"Fréquence prédite (lasso) : {freq_hat_nb:.5f}")

Fréquence observée : 0.07064
Fréquence prédite (lasso) : 0.13027


In [22]:

# Calculate the mean of the product of lasso_pred and sev_hat
result = (lasso_pred * df_test["sev_hat"]).mean()
print(result)

125.86935136062631


## Approche avec une Quasi-Poisson

In [23]:
# On part de df_glm déjà préparé (Exposure > 0, variables catégorielles, etc.)

formula_freq = (
    "ClaimNb ~ C(DriverAgeClass) + C(CarAgeClass) "
    "+ C(Power) + C(Region) + C(Gas) + C(DensityClass)"
)

# Estimation Poisson avec ajustement de dispersion (quasi-Poisson)
model_qp = smf.glm(
    formula=formula_freq,
    data=df_glm,
    family=sm.families.Poisson(),
    offset=np.log(df_glm["Exposure"])
)

result_qp = model_qp.fit(scale="X2")   # ou scale="pearson"
print(result_qp.summary())

# Les prédictions de fréquence restent les mêmes que le Poisson simple
df_glm["lambda_hat_qp"] = result_qp.predict(df_glm, offset=np.log(df_glm["Exposure"]))

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               395215
Model:                            GLM   Df Residuals:                   395180
Model Family:                 Poisson   Df Model:                           34
Link Function:                    Log   Scale:                          1.7431
Method:                          IRLS   Log-Likelihood:                -36972.
Date:                Thu, 04 Dec 2025   Deviance:                       99098.
Time:                        17:28:20   Pearson chi2:                 6.89e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.002349
Covariance Type:            nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 